# NeuroMoE with a pseudospatial memory

Built from commit `d752206`, pre-registered in `PREREGISTRATION_map.md`. Can memories placed on a map by their
relations - read out by place neurons - infer what was never taught, with one exposure per fact and no gradients?
Rivals: the same model without the map (following stored links) and a transformer and GRU trained on thousands of
these episodes. 10 test seeds; map settings tuned on separate validation seeds.

In [ ]:
import os, torch
os.makedirs("/kaggle/working/neuromoe/tests", exist_ok=True)
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
%%writefile /kaggle/working/neuromoe/neuromoe.py
"""NeuroMoE: a growing, gradient-free sparse mixture of experts.

No backprop and no pretrained weights anywhere. The model starts empty and grows as it is used:

  SparseExpansion  fixed random sparse wiring + k-winners-take-all (fly mushroom body / dentate gyrus). Similar
                   inputs share active units, different inputs barely overlap. Never trained.
  ExpertPool       experts are assemblies with input synapses (a prototype over the sparse code) and output synapses.
                   Routing is lateral inhibition: the best-matching experts win. When nothing matches well enough a
                   new expert is born (neurogenesis) wired to exactly the active units (synaptogenesis); when the
                   winner predicts wrong, the match threshold is raised above it and the search continues (ARTMAP
                   match tracking), so exceptions get their own expert instead of overwriting old ones. Winners drift
                   toward their inputs (Hebbian instar: grow synapses to active units, weaken the rest), weak synapses
                   are pruned, experts that stop winning die.
  WorkingMemory    one-shot associative store written by an error-correcting local rule
                   (M += beta (v - M k) k^T / |k|): the value is present when it is written, so no gradient is needed.
                   Sparse keys are nearly orthogonal, so capacity scales with the code size, not with training. A
                   fixed "time cell" code per step lets it bind items to positions (serial recall, n-back).

Every update is local: it uses only the activity on either side of a synapse plus a target that is physically
present at that moment (a label, the next symbol, a stored value). Nothing is differentiated.
"""
import math

import torch


def _generator(seed):
    return torch.Generator().manual_seed(int(seed))


class SparseExpansion:
    """d_in -> n units, each summing `fan_in` randomly chosen inputs with fixed positive weights; exactly k fire.
    Positive, distinct weights make the code deterministic (no ties). An optional homeostatic bias equalises how often
    each unit fires, so no unit is dead - a local rule on each unit's own firing rate."""

    def __init__(self, d_in, n=2048, k=32, fan_in=8, seed=0, homeostasis=0.0):
        g = _generator(seed)
        self.d_in, self.n, self.k = d_in, n, k
        # every input must reach at least 2k units, or a lone active input (a symbol) cannot fill the k winners and
        # the rest are ties at zero - the same filler units for every symbol, which makes keys collide
        fan_in = max(fan_in, math.ceil(2 * k * d_in / n))
        # connections spread evenly over inputs, then shuffled: no input is under-represented by chance
        wiring = torch.arange(n * fan_in) % d_in
        self.idx = wiring[torch.randperm(n * fan_in, generator=g)].reshape(n, fan_in)
        self.w = 0.5 + torch.rand(n, fan_in, generator=g)
        self.bias = torch.zeros(n)
        self.rate = torch.full((n,), k / n)
        self.homeostasis = homeostasis

    @torch.no_grad()
    def __call__(self, x):
        """x (B, d_in) -> binary code (B, n) with exactly k ones."""
        drive = (x[:, self.idx] * self.w).sum(-1) - self.bias
        code = torch.zeros(x.shape[0], self.n)
        code.scatter_(1, drive.topk(self.k, dim=1).indices, 1.0)
        if self.homeostasis:
            self.rate.mul_(0.99).add_(0.01 * code.mean(0))
            self.bias.add_(self.homeostasis * (self.rate - self.k / self.n))
        return code


class ExpertPool:
    """Neurogenesis mixture of experts over binary sparse codes.

    match(e, x) = (P_e . x) / |x| in [0, 1]: the share of the input's active units that expert e is wired to.
    labels="exact": classification; ARTMAP match tracking recruits a new expert whenever the accepted one predicts
    the wrong label. labels="stochastic": next-symbol prediction; an expert splits (a new, more specific expert is
    born) only when it has seen enough evidence and still gave the observed symbol low probability - so contexts are
    refined exactly where the data demands it (variable-order memory)."""

    def __init__(self, n_code, n_out, vigilance=0.7, k_route=3, temperature=0.05, beta=0.2, prune=0.05,
                 max_experts=4096, labels="exact", split_prob=0.1, split_evidence=4, smoothing=0.1, trust=0.0):
        self.n_code, self.n_out = n_code, n_out
        self.vigilance, self.k_route, self.temperature = vigilance, k_route, temperature
        self.beta, self.prune, self.max_experts = beta, prune, max_experts
        self.labels, self.split_prob, self.split_evidence, self.smoothing = labels, split_prob, split_evidence, smoothing
        # trust > 0: an expert's vote is scaled by evidence / (evidence + trust), so a newborn expert resting on one
        # example (possibly a mislabelled one) is heard weakly until other examples confirm it
        self.trust = trust
        self.P = torch.zeros(0, n_code)  # input synapses (prototype strengths)
        self.O = torch.zeros(0, n_out)   # output synapses (association counts)
        self.last_win = torch.zeros(0)
        self.step = 0
        self.born = self.died = 0

    def __len__(self):
        return self.P.shape[0]

    @torch.no_grad()
    def match(self, code):
        return self.P @ code / code.sum().clamp_min(1)

    @torch.no_grad()
    def predict(self, code, return_match=False):
        """Distribution over outputs: the top k_route experts vote, weighted by match (lateral inhibition)."""
        if not len(self):
            p = torch.full((self.n_out,), 1.0 / self.n_out)
            return (p, 0.0) if return_match else p
        m = self.match(code)
        top = m.topk(min(self.k_route, len(self))).indices
        w = self._weights(m[top], top)
        votes = self.O[top] + self.smoothing
        p = (w[:, None] * votes / votes.sum(1, keepdim=True)).sum(0)
        return (p, m.max().item()) if return_match else p

    def _weights(self, match, idx):
        w = torch.softmax(match / self.temperature, -1)
        if self.trust:
            evidence = self.O[idx].sum(-1)
            w = w * evidence / (evidence + self.trust)
            w = w / w.sum(-1, keepdim=True).clamp_min(1e-12)
        return w

    @torch.no_grad()
    def predict_batch(self, codes):
        """Vectorised predict for many inputs at once: (distributions (B, n_out), best match (B,))."""
        B = codes.shape[0]
        if not len(self):
            return torch.full((B, self.n_out), 1.0 / self.n_out), torch.zeros(B)
        m = codes @ self.P.T / codes.sum(1, keepdim=True).clamp_min(1)
        top = m.topk(min(self.k_route, len(self)), dim=1)
        w = self._weights(top.values, top.indices)
        votes = self.O[top.indices] + self.smoothing
        return (w[..., None] * votes / votes.sum(-1, keepdim=True)).sum(1), m.max(1).values

    @torch.no_grad()
    def _birth(self, code, y):
        if len(self) >= self.max_experts:  # full: the longest-idle expert dies first
            self._kill(self.last_win.argmin())
        self.P = torch.cat([self.P, code[None]])
        o = torch.zeros(1, self.n_out)
        o[0, y] = 1.0
        self.O = torch.cat([self.O, o])
        self.last_win = torch.cat([self.last_win, torch.tensor([float(self.step)])])
        self.born += 1
        return len(self) - 1

    @torch.no_grad()
    def _kill(self, e):
        keep = torch.ones(len(self), dtype=torch.bool)
        keep[e] = False
        self.P, self.O, self.last_win = self.P[keep], self.O[keep], self.last_win[keep]
        self.died += 1

    @torch.no_grad()
    def _reinforce(self, e, code, y):
        """Hebbian instar with synaptic turnover: grow toward active units, weaken the rest, prune weak synapses."""
        self.P[e] += self.beta * (code - self.P[e])
        self.P[e][self.P[e] < self.prune] = 0.0
        self.O[e, y] += 1.0
        self.last_win[e] = float(self.step)

    @torch.no_grad()
    def learn(self, code, y):
        """One labelled example, seen once. Returns the expert that learned it."""
        self.step += 1
        if not len(self):
            return self._birth(code, y)
        m = self.match(code)
        rho = self.vigilance
        for e in m.argsort(descending=True).tolist():
            if m[e] < rho:
                break  # nothing left clears the (possibly raised) vigilance
            if self.labels == "exact":
                if self.O[e].argmax().item() == y:
                    self._reinforce(e, code, y)
                    return e
                rho = m[e].item() + 1e-6  # match tracking: this expert is wrong here, demand a closer one
                continue
            # stochastic: accept the best match unless it has confidently learned that y is unlikely here
            counts = self.O[e]
            p_y = (counts[y] + self.smoothing) / (counts.sum() + self.smoothing * self.n_out)
            if counts.sum() >= self.split_evidence and p_y < self.split_prob and m[e] < 0.999:
                break  # a surprise in a context it knows well: split, give this context its own expert
            self._reinforce(e, code, y)
            return e
        return self._birth(code, y)

    def synapses(self):
        return int((self.P > 0).sum()) + int((self.O > 0).sum())


class WorkingMemory:
    """One-shot associative store: M (n_value x n_key), written M += beta (v - M k) k^T / |k|. After a write with
    beta = 1 the key recalls its value exactly; earlier bindings are disturbed only through shared active units.
    `decay` < 1 lets old items fade (a leaky buffer); 1 keeps everything until overwritten."""

    def __init__(self, n_key, n_value, beta=1.0, decay=1.0):
        self.M = torch.zeros(n_value, n_key)
        self.beta, self.decay = beta, decay

    @torch.no_grad()
    def write(self, key, value):
        if self.decay < 1.0:
            self.M.mul_(self.decay)
        self.M += self.beta * torch.outer(value - self.M @ key, key) / key.sum().clamp_min(1)

    @torch.no_grad()
    def read(self, key):
        return self.M @ key

    def clear(self):
        self.M.zero_()

    def synapses(self):
        return int((self.M != 0).sum())


class TimeCells:
    """A fixed sparse code for every time step (hippocampal time cells): step t has its own random k-subset of n
    units, independent of content, so items can be bound to *when* they arrived and replayed in order."""

    def __init__(self, n=2048, k=32, seed=1, max_steps=4096):
        g = _generator(seed)
        self.codes = torch.zeros(max_steps, n)
        for t in range(max_steps):
            self.codes[t, torch.randperm(n, generator=g)[:k]] = 1.0

    def __call__(self, t):
        return self.codes[t]


class NeuroMoE:
    """The assembled model for a symbolic or vector stream: expansion (+ optional context trace) -> expert pool,
    with a working memory alongside. `context` > 0 feeds a decaying trace of recent inputs into the expansion, so the
    same symbol after different histories lands on different experts."""

    def __init__(self, d_in, n_out, n=2048, k=32, fan_in=8, context=0.0, context_decay=0.5, seed=0, **pool):
        self.d_in, self.context, self.context_decay = d_in, context, context_decay
        self.expand = SparseExpansion(d_in * (2 if context else 1), n=n, k=k, fan_in=fan_in, seed=seed)
        self.pool = ExpertPool(n, n_out, **pool)
        self.trace = torch.zeros(d_in)

    @torch.no_grad()
    def encode(self, x):
        """x (d_in,) -> sparse code (n,). With context on, the current input is concatenated with the trace."""
        inp = torch.cat([x, self.context * self.trace]) if self.context else x
        return self.expand(inp[None])[0]

    @torch.no_grad()
    def advance(self, x):
        if self.context:
            self.trace.mul_(self.context_decay).add_(x)

    def reset(self):
        self.trace.zero_()

    def footprint(self):
        """Stored synapses (nonzero weights) - the model's actual size, which grows with use."""
        return self.pool.synapses()

In [ ]:
%%writefile /kaggle/working/neuromoe/cogmap.py
"""CognitiveMap: pseudospatial memory. Items are placed in a low-dimensional latent space by their *relations*, not by
how they look, so geometry carries meaning and relations that were never observed can be computed.

  place neurons   a population tiling the latent space; an item's location is a bump of activity over them, and
                  content is bound to location (Huber: a place cell is a what+where conjunction), so memories can be
                  retrieved *by position* ("what is two steps above X?")
  relations       each relation is a displacement ("taller than" = one step along some direction), learned from use
  one-shot        a new item is placed the moment its first relation arrives: position = known item + displacement
  re-anchoring    a fact that links two separate islands of knowledge shifts the whole smaller island so the fact
                  holds, then merges them (remapping: known memories adapted into one frame)
  consolidation   every observed relation is kept as an episode; replaying episodes applies a local delta rule that
                  moves both items (and the relation vector) toward consistency - scattered facts settle into one map,
                  and a mislabelled fact is outvoted by its neighbours instead of breaking a chain

No gradients: the only update is error = (pos[b] - pos[a]) - d_rel, applied to the three quantities involved.
"""
import math
import random

import torch


class PlaceCells:
    """Place neurons with Gaussian fields on a regular lattice over [-span, span]^dim."""

    def __init__(self, dim=2, per_dim=48, span=24.0, width=0.6):
        axes = torch.linspace(-span, span, per_dim)
        grids = torch.meshgrid(*[axes] * dim, indexing="ij")
        self.centers = torch.stack([g.reshape(-1) for g in grids], 1)
        self.width = width * (2 * span / (per_dim - 1))

    def encode(self, pos):
        """Activity of every place neuron for positions (..., dim)."""
        d2 = ((pos[..., None, :] - self.centers) ** 2).sum(-1)
        return torch.exp(-d2 / (2 * self.width ** 2))


class CognitiveMap:
    def __init__(self, dim=2, lr=0.5, rel_lr=0.1, replay=30, robust=None, place=None, seed=0):
        self.dim, self.lr, self.rel_lr, self.replay = dim, lr, rel_lr, replay
        # robust: the push from any one fact saturates at this size, so a single contradicted fact is outvoted as an
        # outlier instead of dragging its whole neighbourhood (a quadratic rule spreads every error)
        self.robust = robust
        self.rng = random.Random(seed)
        self.gen = torch.Generator().manual_seed(seed)
        self.names, self.index = [], {}
        self.pos = torch.zeros(0, dim)
        self.placed = torch.zeros(0, dtype=torch.bool)
        self.island = []  # which connected island of knowledge each item belongs to
        self.rel = {}
        self.episodes = []
        self.place = place or PlaceCells(dim=dim)
        self._islands = 0

    # ---------------------------------------------------------------- learning
    def _item(self, name):
        if name not in self.index:
            self.index[name] = len(self.names)
            self.names.append(name)
            self.pos = torch.cat([self.pos, torch.zeros(1, self.dim)])
            self.placed = torch.cat([self.placed, torch.tensor([False])])
            self.island.append(-1)
        return self.index[name]

    def _relation(self, rel):
        if rel not in self.rel:
            # a new relation gets its own direction, as orthogonal as possible to the ones already known
            d = torch.randn(self.dim, generator=self.gen)
            for other in self.rel.values():
                d = d - (d @ other) / (other @ other) * other
            if d.norm() < 1e-3:
                d = torch.randn(self.dim, generator=self.gen)
            self.rel[rel] = d / d.norm()
        return self.rel[rel]

    def _fresh_spot(self):
        """An unconnected fact starts its own island, a few steps from the known map, until something links it."""
        self._islands += 1
        direction = torch.randn(self.dim, generator=self.gen)
        return self._centroid() + 4.0 * direction / direction.norm()

    def _centroid(self):
        return self.pos[self.placed].mean(0) if bool(self.placed.any()) else torch.zeros(self.dim)

    def recenter(self):
        """Shift the whole map so its centre sits in the middle of the place-cell lattice. A global shift changes
        no relation; it keeps the map inside the region the place neurons cover (place cells remap to the space in
        use)."""
        if bool(self.placed.any()):
            self.pos[self.placed] -= self._centroid()

    def _consolidate(self, a, rel, b, lr):
        d = self.rel[rel]
        err = (self.pos[b] - self.pos[a]) - d
        if self.robust:
            err = err * min(1.0, self.robust / (float(err.norm()) + 1e-9))
        self.pos[a] += 0.5 * lr * err
        self.pos[b] -= 0.5 * lr * err
        if self.rel_lr:  # the relation's displacement is learned too, but slowly
            step = self.rel_lr * err
            self.rel[rel] = d + step if (d + step).norm() > 0.2 else d

    def observe(self, a, rel, b):
        """One fact: `b` lies one `rel`-step from `a` (e.g. a='Bob', rel='taller', b='Alice' means Alice is taller)."""
        ia, ib, d = self._item(a), self._item(b), self._relation(rel)
        if not self.placed[ia] and not self.placed[ib]:
            self.pos[ia] = self._fresh_spot()
            self.pos[ib] = self.pos[ia] + d
            self.island[ia] = self.island[ib] = self._islands
        elif not self.placed[ib]:
            self.pos[ib] = self.pos[ia] + d
            self.island[ib] = self.island[ia]
        elif not self.placed[ia]:
            self.pos[ia] = self.pos[ib] - d
            self.island[ia] = self.island[ib]
        elif self.island[ia] != self.island[ib]:
            self._reanchor(ia, ib, d)
        else:
            self._consolidate(ia, rel, ib, self.lr)
        self.placed[ia] = self.placed[ib] = True
        self.episodes.append((ia, rel, ib))
        for _ in range(self.replay):  # replay: consolidation over remembered episodes
            ea, er, eb = self.episodes[self.rng.randrange(len(self.episodes))]
            self._consolidate(ea, er, eb, self.lr)

    def _reanchor(self, ia, ib, d):
        """Two islands just got linked: move the smaller one rigidly so the new fact holds, and merge them."""
        members = lambda k: [i for i, isl in enumerate(self.island) if isl == k]
        A, B = members(self.island[ia]), members(self.island[ib])
        shift = (self.pos[ia] + d) - self.pos[ib]
        if len(B) <= len(A):
            self.pos[B] += shift
            keep, move = self.island[ia], B
        else:
            self.pos[A] -= shift
            keep, move = self.island[ib], A
        for i in move:
            self.island[i] = keep

    def sleep(self, rounds=200, damping=0.5, tolerance=0.5):
        """Offline consolidation: every remembered episode is replayed at once (like sleep ripples) and every item
        moves toward where its neighbours, taken together, say it should be. Local - each item hears only its own
        neighbours - but the network relaxes as a whole, so a region misplaced by one bad fact is pulled back as a
        unit. Facts that disagree with the consensus by more than `tolerance` are down-weighted (robust: a lone
        mislabelled fact is outvoted rather than averaged in). Relation vectors settle to the average displacement
        of the facts that carry them."""
        if not self.episodes:
            return
        self.recenter()
        a = torch.tensor([e[0] for e in self.episodes])
        b = torch.tensor([e[2] for e in self.episodes])
        rels = sorted(self.rel)
        r = torch.tensor([rels.index(e[1]) for e in self.episodes])
        for _ in range(rounds):
            D = torch.stack([self.rel[k] for k in rels])[r]
            res = (self.pos[b] - self.pos[a]) - D
            w = torch.clamp(tolerance / (res.norm(dim=1) + 1e-9), max=1.0)[:, None]  # Huber weights
            num = torch.zeros_like(self.pos)
            den = torch.zeros(len(self.pos), 1)
            num.index_add_(0, b, w * (self.pos[a] + D))
            num.index_add_(0, a, w * (self.pos[b] - D))
            den.index_add_(0, b, w)
            den.index_add_(0, a, w)
            has = den[:, 0] > 0
            self.pos[has] = (1 - damping) * self.pos[has] + damping * num[has] / den[has]
            if self.rel_lr:
                disp = self.pos[b] - self.pos[a]
                for k, name in enumerate(rels):
                    sel = r == k
                    avg = (w[sel] * disp[sel]).sum(0) / w[sel].sum().clamp_min(1e-9)
                    self.rel[name] = (1 - self.rel_lr) * self.rel[name] + self.rel_lr * avg

    # ---------------------------------------------------------------- reasoning
    def knows(self, name):
        return name in self.index and bool(self.placed[self.index[name]])

    def steps(self, a, rel, b):
        """How many `rel`-steps b lies from a (positive: b is `rel` of a). The magnitude doubles as confidence -
        which is why far-apart items are compared more reliably (the symbolic distance effect)."""
        d = self.rel[rel]
        return float((self.pos[self.index[b]] - self.pos[self.index[a]]) @ d / (d @ d))

    def at(self, a, moves, top=3):
        """Items nearest to the position reached from `a` by the given moves [(rel, count), ...] - retrieval by
        location through the place neurons, not by any stored link."""
        target = self.pos[self.index[a]].clone()
        for rel, count in moves:
            target += count * self.rel[rel]
        return self.nearest(target, top)

    def nearest(self, target, top=3):
        """Place-cell readout: the items whose place codes overlap most with the code of the target position. For
        Gaussian place fields tiling the space, that overlap falls monotonically with distance (the product of two
        Gaussian bumps integrates to a Gaussian of their separation), so ranking by distance is the exact readout -
        and it keeps working at the lattice's edge, where individual place cells run out."""
        live = self.placed.nonzero().squeeze(1)
        if not len(live):
            return []
        dist = (self.pos[live] - target).norm(dim=1)
        order = dist.argsort()[:top]
        return [(self.names[int(live[i])], float(dist[i])) for i in order]

    def between(self, a, b, rel):
        """Items whose position along `rel` lies strictly between a and b."""
        lo, hi = sorted((0.0, self.steps(a, rel, b)))
        return [n for n in self.names if n not in (a, b) and self.knows(n) and lo + 0.5 < self.steps(a, rel, n) < hi - 0.5]

    def ranking(self, rel):
        """Every placed item ordered along `rel`."""
        d = self.rel[rel]
        live = [n for n in self.names if self.knows(n)]
        return sorted(live, key=lambda n: float(self.pos[self.index[n]] @ d))

    def footprint(self):
        return int(self.placed.sum()) * self.dim + len(self.rel) * self.dim

In [ ]:
%%writefile /kaggle/working/neuromoe/map_bench.py
"""Pseudospatial memory vs memory-only chaining vs trained sequence models (PREREGISTRATION_map.md).

  transitive  N items taught only through adjacent pairs ("hi is above lo"), shuffled, each shown once, a fraction
              `noise` of them mislabelled (reversed). Test: every non-adjacent pair, by symbolic distance.
  grid2d      a W x W grid of items taught only through neighbour relations (east / north), shuffled, once, some
              mislabelled. Test: "from X, move dx east and dy north - which item is there?" for |dx|+|dy| >= 2, i.e.
              places never related to X directly.

  map        cogmap.CognitiveMap: relations as displacements, one-shot placement, re-anchoring, replay (no gradients)
  chain      NeuroMoE working memory only: one-shot bindings per relation, answered by following links (no map)
  gru / transformer   trained by backprop on thousands of such episodes, answering in context from the same facts

  python map_bench.py --seeds 10 --steps 6000 [--device cuda]
"""
import argparse
import json
import math
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from cogmap import CognitiveMap
from neuromoe import SparseExpansion, WorkingMemory

HERE = os.path.dirname(os.path.abspath(__file__))
SYMBOLS = 200


# ------------------------------------------------------------------------------------------------------ tasks
def transitive_episode(N, noise, rng):
    """Returns (items low->high, facts [(lo, hi)] as shown - possibly reversed -, queries [(x, y, y_above_x, dist)])."""
    items = rng.sample(range(SYMBOLS), N)
    facts = []
    for k in range(N - 1):
        lo, hi = items[k], items[k + 1]
        facts.append((hi, lo) if rng.random() < noise else (lo, hi))
    rng.shuffle(facts)
    queries = [(items[i], items[j], j > i, abs(i - j)) for i in range(N) for j in range(N) if abs(i - j) >= 2]
    return items, facts, queries


def grid_episode(W, noise, rng):
    """Returns (grid[x][y] items, facts [(a, rel, b)] meaning b is one `rel` step from a, queries [(X, dx, dy, answer)])."""
    ids = rng.sample(range(SYMBOLS), W * W)
    grid = [[ids[x * W + y] for y in range(W)] for x in range(W)]
    facts = []
    for x in range(W):
        for y in range(W):
            for rel, (nx, ny) in (("east", (x + 1, y)), ("north", (x, y + 1))):
                if nx < W and ny < W:
                    a, b = grid[x][y], grid[nx][ny]
                    facts.append((b, rel, a) if rng.random() < noise else (a, rel, b))
    rng.shuffle(facts)
    queries = [(grid[x][y], tx - x, ty - y, grid[tx][ty]) for x in range(W) for y in range(W)
               for tx in range(W) for ty in range(W) if abs(tx - x) + abs(ty - y) >= 2]
    return grid, facts, queries


# ------------------------------------------------------------------------------------------------------ learners
def new_map(cfg, seed):
    return CognitiveMap(dim=2, lr=cfg["lr"], rel_lr=cfg["rel_lr"], replay=cfg["replay"], robust=cfg["robust"],
                        seed=seed)


def map_transitive(facts, queries, cfg, seed):
    m = new_map(cfg, seed)
    for lo, hi in facts:
        m.observe(str(lo), "above", str(hi))
    m.sleep(cfg["sleep"], tolerance=cfg["tol"])
    return [(m.steps(str(x), "above", str(y)) > 0) == above for x, y, above, _ in queries]


def map_grid(facts, queries, cfg, seed):
    m = new_map(cfg, seed)
    for a, rel, b in facts:
        m.observe(str(a), rel, str(b))
    m.sleep(cfg["sleep"], tolerance=cfg["tol"])
    out = []
    for X, dx, dy, ans in queries:
        best = m.at(str(X), [("east", dx), ("north", dy)], top=1)
        out.append(bool(best) and best[0][0] == str(ans))
    return out


class ChainMemory:
    """Memory-only NeuroMoE: each relation is a one-shot associative store (sparse key -> next item), in both
    directions. Questions are answered by following stored links, one recall at a time."""

    def __init__(self, relations, n=2048, k=32, seed=0):
        self.codes = SparseExpansion(SYMBOLS, n=n, k=k, seed=seed)(torch.eye(SYMBOLS))
        self.eye = torch.eye(SYMBOLS + 1)  # the extra slot means "nothing stored here"
        self.mem = {(r, s): WorkingMemory(n, SYMBOLS + 1) for r in relations for s in (1, -1)}
        self.known = {key: set() for key in self.mem}

    def observe(self, a, rel, b):
        self.mem[(rel, 1)].write(self.codes[a], self.eye[b])
        self.mem[(rel, -1)].write(self.codes[b], self.eye[a])
        self.known[(rel, 1)].add(a)
        self.known[(rel, -1)].add(b)

    def step(self, x, rel, sign):
        if x not in self.known[(rel, sign)]:
            return None
        return int(self.mem[(rel, sign)].read(self.codes[x]).argmax())

    def walk(self, x, rel, sign, n):
        for _ in range(n):
            x = self.step(x, rel, sign)
            if x is None:
                return None
        return x


def chain_transitive(facts, queries, N, rng):
    c = ChainMemory(["above"])
    for lo, hi in facts:
        c.observe(lo, "above", hi)
    out = []
    for x, y, above, _ in queries:
        verdict = None
        for sign in (1, -1):  # search upward, then downward, following links
            z = x
            for _ in range(N):
                z = c.step(z, "above", sign)
                if z is None:
                    break
                if z == y:
                    verdict = sign == 1
                    break
            if verdict is not None:
                break
        out.append((verdict if verdict is not None else rng.random() < 0.5) == above)
    return out


def chain_grid(facts, queries):
    c = ChainMemory(["east", "north"])
    for a, rel, b in facts:
        c.observe(a, rel, b)
    out = []
    for X, dx, dy, ans in queries:
        z = c.walk(X, "east", 1 if dx > 0 else -1, abs(dx))
        z = c.walk(z, "north", 1 if dy > 0 else -1, abs(dy)) if z is not None else None
        out.append(z == ans)
    return out


# ------------------------------------------------------------------------------------------------------ trained baselines
QUERY, ABOVE_TOK = SYMBOLS, SYMBOLS + 1
EAST, NORTH, ASK = SYMBOLS, SYMBOLS + 1, SYMBOLS + 2
DX0, DY0 = SYMBOLS + 3 + 4, SYMBOLS + 12 + 4  # offset tokens for -4..4
VOCAB = SYMBOLS + 21


class TinyTransformer(nn.Module):
    def __init__(self, n_out, d=128, layers=4, heads=4, max_len=768, dropout=0.0):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d)
        self.pos = nn.Parameter(0.02 * torch.randn(max_len, d))
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu", batch_first=True,
                                           norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = nn.Linear(d, n_out)

    def forward(self, tok):
        L = tok.shape[1]
        mask = torch.triu(torch.full((L, L), float("-inf"), device=tok.device), 1)
        return self.out(self.blocks(self.emb(tok) + self.pos[:L], mask=mask, is_causal=True))


class GRUNet(nn.Module):
    def __init__(self, n_out, hidden=256, emb=64):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, emb)
        self.gru = nn.GRU(emb, hidden, batch_first=True)
        self.out = nn.Linear(hidden, n_out)

    def forward(self, tok):
        return self.out(self.gru(self.emb(tok))[0])


def transitive_tokens(facts, queries, max_q=None, rng=None):
    tok = [t for lo, hi in facts for t in (lo, hi)]
    tgt = [-1] * len(tok)
    qs = queries if max_q is None else rng.sample(queries, min(max_q, len(queries)))
    for x, y, above, _ in qs:
        tok += [QUERY, x, y]
        tgt += [-1, -1, int(above)]
    return tok, tgt, qs


def grid_tokens(facts, queries, max_q=None, rng=None):
    tok = [t for a, rel, b in facts for t in (a, EAST if rel == "east" else NORTH, b)]
    tgt = [-1] * len(tok)
    qs = queries if max_q is None else rng.sample(queries, min(max_q, len(queries)))
    for X, dx, dy, ans in qs:
        tok += [ASK, X, DX0 + dx, DY0 + dy]
        tgt += [-1, -1, -1, ans]
    return tok, tgt, qs


def batchify(seqs, device):
    L = max(len(t) for t, _ in seqs)
    tok = torch.full((len(seqs), L), 0, dtype=torch.long)
    tgt = torch.full((len(seqs), L), -1, dtype=torch.long)
    for i, (t, g) in enumerate(seqs):
        tok[i, :len(t)] = torch.tensor(t)
        tgt[i, :len(g)] = torch.tensor(g)
    return tok.to(device), tgt.to(device)


def train_net(net, make_seq, steps, lr, device, batch=32):
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=max(steps, 1), pct_start=0.05)
    net.train()
    for _ in range(steps):
        tok, tgt = batchify([make_seq() for _ in range(batch)], device)
        loss = F.cross_entropy(net(tok).reshape(-1, net.out.out_features), tgt.reshape(-1), ignore_index=-1)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
    net.eval()
    return net


@torch.no_grad()
def net_answers(net, to_tokens, facts, queries, device, chunk=24):
    """The same facts, then queries in chunks of the size seen in training; answers in query order."""
    hits = []
    for i in range(0, len(queries), chunk):
        tok, tgt, _ = to_tokens(facts, queries[i:i + chunk])
        t, g = batchify([(tok, tgt)], device)
        pred = net(t).argmax(-1)[0]
        mask = g[0] >= 0
        hits += (pred[mask] == g[0][mask]).cpu().tolist()
    return hits


# ------------------------------------------------------------------------------------------------------ bench
MAP_GRID = [dict(lr=lr, rel_lr=rl, replay=rp, robust=rb, sleep=sl, tol=tol) for lr in (0.3, 0.6)
            for rl in (0.0, 0.05) for rp in (10, 40) for rb in (None, 0.3) for sl in (100, 300) for tol in (0.3, 0.6)]
NOISES, T_SIZES, W_SIZES = (0.0, 0.05, 0.1), (8, 16, 32), (4, 5)


def by_distance(hits, queries):
    d = {}
    for h, q in zip(hits, queries):
        d.setdefault(q[3], []).append(h)
    return {k: float(np.mean(v)) for k, v in sorted(d.items())}


def slope(acc_by_dist):
    xs = np.array(list(acc_by_dist.keys()), dtype=float)
    ys = np.array(list(acc_by_dist.values()))
    return float(np.polyfit(xs, ys, 1)[0]) if len(xs) > 1 else 0.0


def tune_map(val_seeds=range(1000, 1008)):
    """Map settings chosen on validation episodes only (different seeds from every reported episode), on both tasks."""
    def score(cfg):
        accs = []
        for s in val_seeds:
            rng = random.Random(s)
            _, facts, qs = transitive_episode(16, 0.1, rng)
            accs.append(np.mean(map_transitive(facts, qs, cfg, s)))
            _, gfacts, gqs = grid_episode(5, 0.1, rng)
            accs.append(np.mean(map_grid(gfacts, gqs, cfg, s)))
        return float(np.mean(accs))
    scored = [(score(c), c) for c in MAP_GRID]
    best = max(scored, key=lambda sc: sc[0])
    print(f"map tuned on validation: {best[1]} (val acc {best[0]:.3f})", flush=True)
    return best[1]


def train_baselines(steps, device, lr=1e-3):
    """Transformer and GRU trained on fresh episodes of every size and noise level used in testing."""
    rng = random.Random(12345)
    nets = {}
    for name, make in (("transformer", lambda n: TinyTransformer(n)), ("gru", lambda n: GRUNet(n))):
        t0 = time.time()
        tnet = train_net(make(2).to(device), lambda: transitive_tokens(*transitive_episode(
            rng.choice(range(6, 33)), rng.choice(NOISES), rng)[1:], max_q=24, rng=rng)[:2], steps, lr, device)
        gnet = train_net(make(SYMBOLS).to(device), lambda: grid_tokens(*grid_episode(
            rng.choice((3, 4, 5)), rng.choice(NOISES), rng)[1:], max_q=24, rng=rng)[:2], steps, lr, device)
        nets[name] = (tnet, gnet)
        print(f"trained {name} ({steps} steps each task) in {time.time() - t0:.0f}s", flush=True)
    return nets


def run(seeds, steps, device):
    cfg = tune_map()
    nets = train_baselines(steps, device) if steps else {}
    rows = []
    for s in range(seeds):
        row = {"seed": s}
        for noise in NOISES:
            for N in T_SIZES:
                rng = random.Random(10_000 * s + int(noise * 100) + N)
                _, facts, qs = transitive_episode(N, noise, rng)
                results = {"map": map_transitive(facts, qs, cfg, s), "chain": chain_transitive(facts, qs, N, rng)}
                for name, (tnet, _) in nets.items():
                    results[name] = net_answers(tnet, transitive_tokens, facts, qs, device)
                for name, hits in results.items():
                    tag = f"transitive_N{N}_noise{noise}_{name}"
                    dist = by_distance(hits, qs)
                    row[f"{tag}_acc"] = float(np.mean(hits))
                    row[f"{tag}_slope"] = slope(dist)
                    row[f"{tag}_bydist"] = dist
            for W in W_SIZES:
                rng = random.Random(20_000 * s + int(noise * 100) + W)
                _, facts, qs = grid_episode(W, noise, rng)
                results = {"map": map_grid(facts, qs, cfg, s), "chain": chain_grid(facts, qs)}
                for name, (_, gnet) in nets.items():
                    results[name] = net_answers(gnet, grid_tokens, facts, qs, device)
                for name, hits in results.items():
                    row[f"grid_W{W}_noise{noise}_{name}_acc"] = float(np.mean(hits))
        rows.append(row)
        print(f"seed {s} done", flush=True)
    return rows, cfg


PRIMARY = [  # id, metric key template, A, B  (paired over seeds, two-sided, Holm across all four)
    ("M1", "transitive_N16_noise0.1_{}_acc", "map", "chain"),
    ("M2", "transitive_N16_noise0.1_{}_slope", "map", "chain"),
    ("M3", "transitive_N16_noise0.1_{}_acc", "map", "transformer"),
    ("M4", "grid_W5_noise0.1_{}_acc", "map", "chain"),
]


def analyze(rows):
    from scipy import stats
    out, ps = ["# Pseudospatial memory: pre-registered results (PREREGISTRATION_map.md)", "",
               "| id | comparison | n | mean diff | 95% CI | p | p (Holm) | verdict |", "|---|---|---|---|---|---|---|---|"], []
    stats_rows = []
    for pid, key, a, b in PRIMARY:
        d = np.array([r[key.format(a)] - r[key.format(b)] for r in rows if key.format(a) in r and key.format(b) in r])
        if len(d) > 1 and d.std() > 0:
            p = stats.ttest_1samp(d, 0.0).pvalue
            half = stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / math.sqrt(len(d))
        else:
            p, half = (1.0 if len(d) < 2 else (0.0 if d.mean() != 0 else 1.0)), 0.0
        ps.append(p)
        stats_rows.append((pid, key.format(f"{a} - {b}"), len(d), d.mean() if len(d) else float("nan"), half, p))
    order, running, adj = np.argsort(ps), 0.0, [0.0] * len(ps)
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = running
    for (pid, comp, n, mean, half, p), pa in zip(stats_rows, adj):
        out.append(f"| {pid} | {comp} | {n} | {mean:+.4f} | ±{half:.4f} | {p:.3g} | {pa:.3g} | "
                   f"{'significant' if pa < 0.05 else 'not significant'} |")
    out += ["", "## All conditions (mean over seeds)", ""]
    keys = sorted({k for r in rows for k in r if k.endswith(("_acc", "_slope"))})
    for k in keys:
        out.append(f"- `{k}`: {np.mean([r[k] for r in rows if k in r]):.4f}")
    return "\n".join(out)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--seeds", type=int, default=10)
    ap.add_argument("--steps", type=int, default=6000, help="training steps per baseline per task (0: skip them)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--out", default=os.path.join(HERE, "results", "map"))
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    rows, cfg = run(args.seeds, args.steps, args.device)
    json.dump({"args": vars(args), "map_config": cfg, "results": rows},
              open(os.path.join(args.out, f"map_{time.strftime('%Y%m%d-%H%M%S')}.json"), "w"), indent=1)
    # the "trained model" the chat interface loads: tuned settings only - it starts with no knowledge
    json.dump({"model": "NeuroMoE cognitive map", "map_config": cfg, "place_cells": {"per_dim": 48, "span": 24.0},
               "note": "no stored knowledge: it learns from what it is told"},
              open(os.path.join(args.out, "neuromoe_map_config.json"), "w"), indent=1)
    report = analyze(rows)
    open(os.path.join(args.out, "ANALYSIS_map.md"), "w", encoding="utf-8").write(report + "\n")
    print(report)
    print("saved to", args.out)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_neuromoe.py
"""Correctness checks for NeuroMoE's core promises, before any benchmark."""
import os
import sys

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from neuromoe import ExpertPool, NeuroMoE, SparseExpansion, TimeCells, WorkingMemory

torch.manual_seed(0)
ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  ' + detail if detail else ''}")


V = 64
eye = torch.eye(V)

# 1. sparse expansion: exactly k active, deterministic, similarity-preserving
ex = SparseExpansion(V, n=2048, k=32, seed=0)
c = ex(eye)
check("expansion fires exactly k units per input", bool((c.sum(1) == 32).all()))
check("expansion is deterministic", torch.equal(c, ex(eye)))
x = torch.rand(200, 20)
ex2 = SparseExpansion(20, n=2048, k=32, seed=1)
near, far = x + 0.05 * torch.randn_like(x), torch.rand(200, 20)
ov = lambda a, b: (ex2(a) * ex2(b)).sum(1).mean().item() / 32
check("similar inputs share more active units than unrelated ones", ov(x, near) > 2 * ov(x, far),
      f"{ov(x, near):.2f} vs {ov(x, far):.2f}")

# 2. working memory: one-shot exact recall, and it holds many bindings at once
wm = WorkingMemory(2048, V)
keys = SparseExpansion(1000, n=2048, k=32, seed=2)(torch.eye(1000))
vals = torch.randint(V, (1000,))
wm.write(keys[0], eye[vals[0]])
check("one write, exact recall", torch.allclose(wm.read(keys[0]), eye[vals[0]], atol=1e-5))
wm.clear()
for i in range(300):
    wm.write(keys[i], eye[vals[i]])
acc = (torch.stack([wm.read(keys[i]) for i in range(300)]).argmax(1) == vals[:300]).float().mean().item()
check("300 bindings written once each, recalled", acc > 0.95, f"accuracy {acc:.3f}")

# 3. expert pool: one-shot, match tracking, no forgetting, bounded memory, no gradients
pool = ExpertPool(2048, 10, vigilance=0.5)
codes = SparseExpansion(10, n=2048, k=32, seed=3)(torch.eye(10))
for y in range(10):
    pool.learn(codes[y], y)
check("one example per class, all recalled", all(pool.predict(codes[y]).argmax().item() == y for y in range(10)),
      f"{len(pool)} experts")
n0 = len(pool)
pool.learn(codes[3], 7)  # a conflicting label on an identical input
check("a contradiction recruits a new expert (match tracking)", len(pool) == n0 + 1)
pool2 = ExpertPool(2048, 10, vigilance=0.5)
for y in range(5):
    for _ in range(20):
        pool2.learn(codes[y], y)
for y in range(5, 10):
    for _ in range(20):
        pool2.learn(codes[y], y)
check("learning classes 5-9 does not erase classes 0-4",
      all(pool2.predict(codes[y]).argmax().item() == y for y in range(5)))
small = ExpertPool(2048, 10, vigilance=0.99, max_experts=8)
for i in range(50):
    small.learn(SparseExpansion(50, n=2048, k=32, seed=4)(torch.eye(50))[i], i % 10)
check("memory stays bounded (old experts die)", len(small) == 8 and small.died > 0, f"{small.died} died")
check("no tensor anywhere requires a gradient",
      not any(t.requires_grad for t in (pool.P, pool.O, wm.M, ex.w)))

# 4. variable-order sequence learning: two sequences that share a prefix, differ only in the last symbol
seqs = ["xabcdy", "zabcdw"]
stoi = {ch: i for i, ch in enumerate(sorted(set("".join(seqs))))}
m = NeuroMoE(len(stoi), len(stoi), context=1.0, context_decay=0.8, labels="stochastic", vigilance=0.6,
             split_evidence=1, split_prob=0.5)
onehot = torch.eye(len(stoi))
for _ in range(20):
    for s in seqs:
        m.reset()
        for a, b in zip(s, s[1:]):
            m.pool.learn(m.encode(onehot[stoi[a]]), stoi[b])
            m.advance(onehot[stoi[a]])
correct = 0
for s in seqs:
    m.reset()
    for a in s[:-2]:
        m.advance(onehot[stoi[a]])
    correct += m.pool.predict(m.encode(onehot[stoi[s[-2]]])).argmax().item() == stoi[s[-1]]
check("learns which symbol follows 'abcd' from what came four steps earlier", correct == 2, f"{len(m.pool)} experts")

# 5. time cells: distinct codes per step, so position binding is clean
tc = TimeCells(n=2048, k=32, max_steps=100)
overlap = max((tc(i) * tc(j)).sum().item() for i in range(20) for j in range(20) if i != j)
check("time-cell codes for different steps barely overlap", overlap <= 6, f"max shared units {overlap:.0f} / 32")

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/tests/check_cogmap.py
"""Correctness checks for the pseudospatial memory (cogmap.py)."""
import os
import random
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import map_bench as mb
from cogmap import CognitiveMap

ok = True


def check(name, cond, detail=""):
    global ok
    ok &= bool(cond)
    print(f"[{'PASS' if cond else 'FAIL'}] {name}{'  ' + detail if detail else ''}")


# 1. one-shot placement: the first relation fixes the new item's position exactly
m = CognitiveMap(dim=2)
m.observe("A", "above", "B")
check("one fact places the new item one step away", abs(m.steps("A", "above", "B") - 1.0) < 1e-6)

# 2. transitive inference from adjacent pairs only, shuffled, clean: every unseen pair right, full ranking right
items = [f"i{k}" for k in range(20)]
pairs = [(items[k], items[k + 1]) for k in range(19)]
random.Random(1).shuffle(pairs)
m = CognitiveMap(dim=2, robust=0.3)
for lo, hi in pairs:
    m.observe(lo, "above", hi)
m.sleep(300)
acc = np.mean([(m.steps(items[i], "above", items[j]) > 0) == (j > i)
               for i in range(20) for j in range(20) if abs(i - j) > 1])
check("clean transitive inference on never-seen pairs", acc == 1.0, f"{acc:.3f}")
check("full ranking recovered", m.ranking("above") == items)
check("items between two others are found", set(m.between("i3", "i7", "above")) == {"i4", "i5", "i6"})

# 3. re-centering moves the map but changes no relation
before = m.steps("i2", "above", "i15")
m.recenter()
check("re-centering preserves every relation", abs(m.steps("i2", "above", "i15") - before) < 1e-5)

# 4. retrieval by location on a clean 2-D grid, to places never related directly
rng = random.Random(3)
_, facts, qs = mb.grid_episode(5, 0.0, rng)
g = CognitiveMap(dim=2, robust=0.3)
for a, rel, b in facts:
    g.observe(str(a), rel, str(b))
g.sleep(300)
acc = np.mean([g.at(str(X), [("east", dx), ("north", dy)], top=1)[0][0] == str(ans) for X, dx, dy, ans in qs])
check("clean 2-D grid: every multi-step place found by location", acc == 1.0, f"{acc:.3f}")
check("different relations get (nearly) orthogonal directions",
      abs(float(g.rel["east"] @ g.rel["north"])) / (g.rel["east"].norm() * g.rel["north"].norm()) < 0.2)

# 5. the distance effect: with mislabelled facts, the map's errors fall on near pairs, chaining's on far ones
cfg = dict(lr=0.6, rel_lr=0.0, replay=40, robust=0.3, sleep=300, tol=0.3)
map_slopes, chain_slopes = [], []
for s in range(12):
    r = random.Random(100 + s)
    _, facts, qs = mb.transitive_episode(24, 0.1, r)
    map_slopes.append(mb.slope(mb.by_distance(mb.map_transitive(facts, qs, cfg, s), qs)))
    chain_slopes.append(mb.slope(mb.by_distance(mb.chain_transitive(facts, qs, 24, r), qs)))
check("distance effect: map's accuracy slope above chaining's", np.mean(map_slopes) > np.mean(chain_slopes),
      f"map {np.mean(map_slopes):+.4f} vs chain {np.mean(chain_slopes):+.4f}")

# 6. nothing is trained by gradient
check("no tensor requires a gradient", not m.pos.requires_grad and not any(v.requires_grad for v in m.rel.values()))

print("\nALL PASS" if ok else "\nFAILURES ABOVE")
sys.exit(0 if ok else 1)

In [ ]:
%%writefile /kaggle/working/neuromoe/PREREGISTRATION_map.md
# Pseudospatial memory: pre-registration

Committed before the reported run. Hypothesis (from the user): NeuroMoE hit a memory ceiling because its memories are
related only by how much their inputs overlap. Placing memories in a *pseudospatial* map, positioned by their relations
and read out by place neurons, should allow inference about pairs never experienced together.

## Learners
- **map** — `cogmap.CognitiveMap`: relations as learned displacements, one-shot placement, re-anchoring of linked
  islands, robust replay consolidation, retrieval by location. No gradients.
- **chain** — NeuroMoE working memory only: one-shot bindings per relation, answered by following stored links.
- **transformer**, **gru** — trained by backprop on thousands of episodes of every size and noise level used in
  testing, then answering in context from the same facts.

## Tasks
- **transitive**: N ∈ {8, 16, 32} items taught only through adjacent pairs, shuffled, each shown once; a fraction
  noise ∈ {0, 0.05, 0.1} mislabelled (reversed). Test: every pair at least 2 apart. Also accuracy by symbolic distance
  and its slope (the distance effect: positive means far pairs are easier, as in humans and animals).
- **grid**: W × W items (W ∈ {4, 5}) taught only through east/north neighbour facts, same presentation and noise.
  Test: "from X, move dx east and dy north: which item is there?" for |dx| + |dy| ≥ 2.

## Protocol
Map settings chosen on validation episodes (seeds 1000–1007) only; reported episodes use seeds 0–9. The baselines
are trained on freshly generated episodes. Paired over seeds, two-sided t-tests, Holm–Bonferroni across the four
primary tests, α = 0.05.

## Primary tests
| id | comparison (N = 16 or W = 5, noise = 0.1) | metric |
|---|---|---|
| M1 | map − chain | transitive accuracy on unseen pairs |
| M2 | map − chain | slope of accuracy over distance |
| M3 | map − trained transformer | transitive accuracy on unseen pairs |
| M4 | map − chain | grid accuracy at never-related places |

## Predictions and decision rules
- The hypothesis predicts M1 > 0, M2 > 0 (map positive or flat, chaining negative) and M4 > 0.
- M3 is open: a transformer trained on thousands of these episodes may match or beat the map. If it does, the claim
  narrows to "comparable inference with no training and one exposure per fact".
- If M1 and M4 are not significant, the pseudospatial-map hypothesis is not supported in this setup.

## Exploratory
Every other size and noise level; GRU; the transformer on the grid task.

In [ ]:
import subprocess, sys
for test in ("tests/check_neuromoe.py", "tests/check_cogmap.py"):
    r = subprocess.run([sys.executable, f"/kaggle/working/neuromoe/{test}"], capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
    print(r.stdout[-2500:], r.stderr[-1500:])
    assert r.returncode == 0, f"{test} failed - not running the benchmark"

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-u", "/kaggle/working/neuromoe/map_bench.py", "--seeds", "10", "--steps", "6000",
                    "--out", "/kaggle/working/results/map"], capture_output=True, text=True, cwd="/kaggle/working/neuromoe")
print(r.stdout[-12000:], r.stderr[-3000:])

Verdicts: `results/map/ANALYSIS_map.md`. The config the chat interface loads: `results/map/neuromoe_map_config.json`.